# Churn Prediction with LSTM + Attention
## Using Real Snowflake Data with 12-Month Lookback

This notebook trains a churn prediction model using:
- **Data Source**: Snowflake tables (PHONE_USAGE_DATA, ACCOUNT_ATTRIBUTES_MONTHLY, CHURN_RECORDS)
- **Model**: LSTM with Attention mechanism
- **Features**: 12-month usage sequences
- **Target**: Predict if account will churn

---

## 1. Setup and Imports

In [ ]:
# Core libraries
import warnings
import numpy as np
import pandas as pd
from datetime import datetime
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

# Sklearn
from sklearn.metrics import (
    auc, roc_curve, precision_score, recall_score,
    f1_score, confusion_matrix, precision_recall_curve,
    classification_report
)
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Snowflake
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, lit, count, sum as spark_sum
from snowflake.snowpark.types import StructType, StructField, StringType, FloatType, IntegerType

# Progress bar
from tqdm import tqdm

print("✓ Libraries imported successfully")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Using device: {device}")

## 2. Connect to Snowflake

In [ ]:
# Get active Snowflake session
session = get_active_session()

# Set database and schema to MY_DATABASE.PUBLIC
session.use_database("MY_DATABASE")
session.use_schema("PUBLIC")

print("✓ Snowflake session active")
print(f"  Database: {session.get_current_database()}")
print(f"  Schema: {session.get_current_schema()}")
print(f"  Warehouse: {session.get_current_warehouse()}")
print(f"  Role: {session.get_current_role()}")

# Verify tables exist
print("\n✓ Verifying tables exist...")
tables_to_check = ["PHONE_USAGE_DATA", "ACCOUNT_ATTRIBUTES_MONTHLY", "CHURN_RECORDS"]
for table in tables_to_check:
    count = session.table(f"MY_DATABASE.PUBLIC.{table}").count()
    print(f"  {table}: {count:,} rows")

## 3. Load Data from Snowflake Tables

In [ ]:
# Load data from Snowflake tables in MY_DATABASE.PUBLIC
print("Loading data from MY_DATABASE.PUBLIC...")

# 1. Usage data
usage_df = session.table("MY_DATABASE.PUBLIC.PHONE_USAGE_DATA").to_pandas()
usage_df['MONTH'] = pd.to_datetime(usage_df['MONTH'])
print(f"✓ PHONE_USAGE_DATA: {len(usage_df):,} rows")

# 2. Account attributes
account_df = session.table("MY_DATABASE.PUBLIC.ACCOUNT_ATTRIBUTES_MONTHLY").to_pandas()
account_df['MONTH'] = pd.to_datetime(account_df['MONTH'])
print(f"✓ ACCOUNT_ATTRIBUTES_MONTHLY: {len(account_df):,} rows")

# 3. Churn records (for ground truth labels)
churn_df = session.table("MY_DATABASE.PUBLIC.CHURN_RECORDS").to_pandas()
churn_df['CHURN_DATE'] = pd.to_datetime(churn_df['CHURN_DATE'])
print(f"✓ CHURN_RECORDS: {len(churn_df):,} rows")

print(f"\nData date range: {usage_df['MONTH'].min()} to {usage_df['MONTH'].max()}")
print(f"Unique accounts: {usage_df['USERID'].nunique():,}")
print(f"Churned accounts in CHURN_RECORDS: {len(churn_df):,}")

In [ ]:
# Display data schema
print("\n=== PHONE_USAGE_DATA Schema ===")
print(usage_df.dtypes)

print("\n=== Sample Usage Data ===")
print(usage_df.head())

## 4. Feature Engineering - Create 12-Month Sequences

In [ ]:
def create_churn_sequences(usage_df, account_df, churn_df, max_lookback=12):
    """
    Create time-series sequences for churn prediction.
    
    For each account:
    - Extract up to 12 months of usage history
    - Create feature vectors for each month
    - Label with churn status
    
    Returns:
        DataFrame with columns: account_id, sequence (list of feature vectors), churn_label, seq_length
    """
    print("\n" + "="*70)
    print("Creating 12-Month Sequences for Churn Prediction")
    print("="*70)
    
    # Get list of churned accounts
    churned_accounts = set(churn_df['USERID'].unique())
    print(f"\nChurned accounts: {len(churned_accounts):,}")
    
    # Get all unique accounts
    all_accounts = usage_df['USERID'].unique()
    print(f"Total accounts: {len(all_accounts):,}")
    
    sequences = []
    
    for account_id in tqdm(all_accounts, desc="Processing accounts"):
        # Get usage history for this account
        account_usage = usage_df[usage_df['USERID'] == account_id].sort_values('MONTH')
        
        # Skip if less than 3 months of data
        if len(account_usage) < 3:
            continue
        
        # Take last 12 months (or all available if less)
        account_usage = account_usage.tail(max_lookback)
        
        # Check if churned
        is_churned = 1 if account_id in churned_accounts else 0
        
        # Extract features for each month
        monthly_features = []
        
        for _, row in account_usage.iterrows():
            # Normalize features to [0, 1] range (approximately)
            features = [
                # Call volume features
                min(row['PHONE_TOTAL_CALLS'] / 1000, 1.0),  # Normalized total calls
                min(row['PHONE_TOTAL_MINUTES_OF_USE'] / 10000, 1.0),  # Normalized minutes
                min(row['VOICE_CALLS'] / 1000, 1.0),  # Voice calls
                min(row['FAX_CALLS'] / 100, 1.0),  # Fax calls
                
                # Call direction features
                min(row['PHONE_TOTAL_NUM_INBOUND_CALLS'] / 500, 1.0),
                min(row['PHONE_TOTAL_NUM_OUTBOUND_CALLS'] / 500, 1.0),
                
                # Device usage features
                min(row['HARDPHONE_CALLS'] / 500, 1.0),
                min(row['SOFTPHONE_CALLS'] / 500, 1.0),
                min(row['MOBILE_CALLS'] / 500, 1.0),
                
                # Engagement feature
                min(row['PHONE_MAU'] / 100, 1.0),  # Monthly active users
            ]
            
            monthly_features.append(features)
        
        sequences.append({
            'account_id': account_id,
            'sequence': monthly_features,
            'churn_label': is_churned,
            'seq_length': len(monthly_features)
        })
    
    # Create DataFrame
    result_df = pd.DataFrame(sequences)
    
    print(f"\n✓ Created {len(result_df):,} sequences")
    print(f"  Churn rate: {result_df['churn_label'].mean():.2%}")
    print(f"  Avg sequence length: {result_df['seq_length'].mean():.1f} months")
    print(f"  Min/Max length: {result_df['seq_length'].min()}/{result_df['seq_length'].max()} months")
    
    # Show feature statistics
    print(f"\n  Feature vector size: {len(monthly_features[0])}")
    print(f"  Features:")
    feature_names = [
        'Total Calls (norm)', 'Total Minutes (norm)', 'Voice Calls (norm)', 'Fax Calls (norm)',
        'Inbound Calls (norm)', 'Outbound Calls (norm)',
        'Hardphone (norm)', 'Softphone (norm)', 'Mobile (norm)',
        'MAU (norm)'
    ]
    for i, name in enumerate(feature_names):
        print(f"    [{i}] {name}")
    
    return result_df

# Create sequences
sequence_df = create_churn_sequences(usage_df, account_df, churn_df, max_lookback=12)

In [ ]:
def create_churn_sequences(usage_df, account_df, churn_df, max_lookback=12):
    """
    Create time-series sequences for churn prediction.
    
    For each account:
    - Extract up to 12 months of usage history
    - Create feature vectors for each month
    - Label with churn status from CHURN_RECORDS table
    
    Returns:
        DataFrame with columns: account_id, sequence (list of feature vectors), churn_label, seq_length
    """
    print("\n" + "="*70)
    print("Creating 12-Month Sequences for Churn Prediction")
    print("="*70)
    
    # Get list of churned accounts from CHURN_RECORDS
    churned_accounts = set(churn_df['USERID'].unique())
    print(f"\nChurned accounts from CHURN_RECORDS: {len(churned_accounts):,}")
    
    # Get all unique accounts
    all_accounts = usage_df['USERID'].unique()
    print(f"Total accounts: {len(all_accounts):,}")
    
    sequences = []
    
    for account_id in tqdm(all_accounts, desc="Processing accounts"):
        # Get usage history for this account
        account_usage = usage_df[usage_df['USERID'] == account_id].sort_values('MONTH')
        
        # Skip if less than 3 months of data
        if len(account_usage) < 3:
            continue
        
        # Take last 12 months (or all available if less)
        account_usage = account_usage.tail(max_lookback)
        
        # Check if churned (from CHURN_RECORDS table)
        is_churned = 1 if account_id in churned_accounts else 0
        
        # Extract features for each month
        monthly_features = []
        
        for _, row in account_usage.iterrows():
            # Normalize features to [0, 1] range (approximately)
            features = [
                # Call volume features
                min(row['PHONE_TOTAL_CALLS'] / 1000, 1.0),  # Normalized total calls
                min(row['PHONE_TOTAL_MINUTES_OF_USE'] / 10000, 1.0),  # Normalized minutes
                min(row['VOICE_CALLS'] / 1000, 1.0),  # Voice calls
                min(row['FAX_CALLS'] / 100, 1.0),  # Fax calls
                
                # Call direction features
                min(row['PHONE_TOTAL_NUM_INBOUND_CALLS'] / 500, 1.0),
                min(row['PHONE_TOTAL_NUM_OUTBOUND_CALLS'] / 500, 1.0),
                
                # Device usage features
                min(row['HARDPHONE_CALLS'] / 500, 1.0),
                min(row['SOFTPHONE_CALLS'] / 500, 1.0),
                min(row['MOBILE_CALLS'] / 500, 1.0),
                
                # Engagement feature
                min(row['PHONE_MAU'] / 100, 1.0),  # Monthly active users
            ]
            
            monthly_features.append(features)
        
        sequences.append({
            'account_id': account_id,
            'sequence': monthly_features,
            'churn_label': is_churned,
            'seq_length': len(monthly_features)
        })
    
    # Create DataFrame
    result_df = pd.DataFrame(sequences)
    
    print(f"\n✓ Created {len(result_df):,} sequences")
    print(f"  Churn rate: {result_df['churn_label'].mean():.2%}")
    print(f"  Avg sequence length: {result_df['seq_length'].mean():.1f} months")
    print(f"  Min/Max length: {result_df['seq_length'].min()}/{result_df['seq_length'].max()} months")
    
    # Show feature statistics
    print(f"\n  Feature vector size: {len(monthly_features[0])}")
    print(f"  Features:")
    feature_names = [
        'Total Calls (norm)', 'Total Minutes (norm)', 'Voice Calls (norm)', 'Fax Calls (norm)',
        'Inbound Calls (norm)', 'Outbound Calls (norm)',
        'Hardphone (norm)', 'Softphone (norm)', 'Mobile (norm)',
        'MAU (norm)'
    ]
    for i, name in enumerate(feature_names):
        print(f"    [{i}] {name}")
    
    return result_df

# Create sequences using CHURN_RECORDS for labels
sequence_df = create_churn_sequences(usage_df, account_df, churn_df, max_lookback=12)

## 5. Split Data into Train/Val/Test

In [ ]:
def check_data_cleanliness(sequence_df, verbose=True):
    """
    Check if churn patterns in the data are too clean/obvious.
    
    Returns a score from 0-100:
    - 0-30: Realistic, noisy data (good for ML)
    - 30-60: Moderate patterns (acceptable)
    - 60-100: Too clean, patterns too obvious (unrealistic)
    
    Args:
        sequence_df: DataFrame with sequences and churn labels
        verbose: If True, print detailed analysis
    
    Returns:
        dict with cleanliness scores and diagnostics
    """
    import scipy.stats as stats
    
    churned_data = sequence_df[sequence_df['churn_label'] == 1]
    active_data = sequence_df[sequence_df['churn_label'] == 0]
    
    results = {
        'overall_score': 0,
        'feature_separation_score': 0,
        'trend_score': 0,
        'variance_score': 0,
        'details': {}
    }
    
    if verbose:
        print("\n" + "="*70)
        print("DATA CLEANLINESS ANALYSIS")
        print("="*70)
    
    # 1. Feature Separation Analysis
    if verbose:
        print("\n1️⃣ FEATURE SEPARATION (last month)")
        print("-" * 70)
    
    churned_last = np.array([seq[-1] for seq in churned_data['sequence']])
    active_last = np.array([seq[-1] for seq in active_data['sequence']])
    
    feature_names = ['Calls', 'Minutes', 'Voice', 'Fax', 'Inbound', 
                    'Outbound', 'Hardphone', 'Softphone', 'Mobile', 'MAU']
    
    separations = []
    for i, name in enumerate(feature_names):
        churned_vals = churned_last[:, i]
        active_vals = active_last[:, i]
        
        # Calculate separation (normalized difference)
        sep = abs(churned_vals.mean() - active_vals.mean())
        
        # T-test for statistical significance
        t_stat, p_value = stats.ttest_ind(churned_vals, active_vals)
        
        separations.append(sep)
        
        if verbose:
            status = "🔴 TOO OBVIOUS" if sep > 0.3 else "🟡 MODERATE" if sep > 0.15 else "🟢 REALISTIC"
            print(f"  {name:<12} diff={sep:.4f}  p={p_value:.4f}  {status}")
    
    avg_separation = np.mean(separations)
    separation_score = min(100, avg_separation * 200)  # Scale to 0-100
    results['feature_separation_score'] = separation_score
    results['details']['avg_feature_separation'] = avg_separation
    
    if verbose:
        print(f"\n  Average separation: {avg_separation:.4f}")
        print(f"  Separation score: {separation_score:.1f}/100")
    
    # 2. Trend Analysis (declining patterns)
    if verbose:
        print("\n2️⃣ TREND ANALYSIS (usage decline over time)")
        print("-" * 70)
    
    def calculate_trend(sequences):
        """Calculate average trend (slope) across all sequences"""
        trends = []
        for seq in sequences:
            if len(seq) >= 3:
                # Use total calls (feature 0) as proxy
                calls = [month[0] for month in seq]
                x = np.arange(len(calls))
                if len(x) > 1:
                    slope = np.polyfit(x, calls, 1)[0]
                    trends.append(slope)
        return np.array(trends)
    
    churned_trends = calculate_trend(churned_data['sequence'].values)
    active_trends = calculate_trend(active_data['sequence'].values)
    
    trend_diff = abs(churned_trends.mean() - active_trends.mean())
    trend_score = min(100, trend_diff * 500)  # Scale to 0-100
    results['trend_score'] = trend_score
    results['details']['churned_avg_trend'] = churned_trends.mean()
    results['details']['active_avg_trend'] = active_trends.mean()
    
    if verbose:
        print(f"  Churned accounts avg trend: {churned_trends.mean():.6f}")
        print(f"  Active accounts avg trend:  {active_trends.mean():.6f}")
        print(f"  Trend difference: {trend_diff:.6f}")
        print(f"  Trend score: {trend_score:.1f}/100")
        
        if trend_diff > 0.05:
            print(f"  🔴 Churned accounts show obvious declining trend!")
        elif trend_diff > 0.02:
            print(f"  🟡 Moderate trend difference")
        else:
            print(f"  🟢 Realistic trend patterns")
    
    # 3. Variance Analysis (data noisiness)
    if verbose:
        print("\n3️⃣ VARIANCE ANALYSIS (data noisiness)")
        print("-" * 70)
    
    churned_var = np.var(churned_last, axis=0).mean()
    active_var = np.var(active_last, axis=0).mean()
    avg_variance = (churned_var + active_var) / 2
    
    # Lower variance = cleaner data = higher score
    variance_score = max(0, 100 - (avg_variance * 500))
    results['variance_score'] = variance_score
    results['details']['avg_variance'] = avg_variance
    
    if verbose:
        print(f"  Average variance: {avg_variance:.6f}")
        print(f"  Variance score: {variance_score:.1f}/100")
        
        if avg_variance < 0.01:
            print(f"  🔴 Very low variance - data too uniform!")
        elif avg_variance < 0.05:
            print(f"  🟡 Moderate variance")
        else:
            print(f"  🟢 High variance - realistic noise")
    
    # 4. Overall Score
    overall_score = (separation_score * 0.5 + trend_score * 0.3 + variance_score * 0.2)
    results['overall_score'] = overall_score
    
    if verbose:
        print("\n" + "="*70)
        print("OVERALL CLEANLINESS SCORE")
        print("="*70)
        print(f"\n  Overall Score: {overall_score:.1f}/100")
        print()
        
        if overall_score > 60:
            print("  🔴 DATA TOO CLEAN - Patterns are too obvious!")
            print("     The model will achieve unrealistically high accuracy.")
            print("     Consider:")
            print("       - Adding noise to features")
            print("       - Making churn patterns more gradual")
            print("       - Using real-world data")
        elif overall_score > 30:
            print("  🟡 MODERATELY CLEAN - Acceptable for testing")
            print("     Patterns are somewhat obvious but usable.")
        else:
            print("  🟢 REALISTIC DATA - Good noise and complexity")
            print("     This data should produce realistic model performance.")
    
    return results

# Run the analysis
cleanliness_results = check_data_cleanliness(sequence_df, verbose=True)

In [ ]:
def split_data(df, train_ratio=0.7, val_ratio=0.15, random_state=42):
    """Split data into train, validation, and test sets"""
    # Shuffle data
    df = df.sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    n = len(df)
    train_size = int(train_ratio * n)
    val_size = int(val_ratio * n)
    
    train_df = df[:train_size].reset_index(drop=True)
    val_df = df[train_size:train_size+val_size].reset_index(drop=True)
    test_df = df[train_size+val_size:].reset_index(drop=True)
    
    print(f"\nData split:")
    print(f"  Train: {len(train_df):,} samples ({len(train_df)/n:.1%})")
    print(f"  Val:   {len(val_df):,} samples ({len(val_df)/n:.1%})")
    print(f"  Test:  {len(test_df):,} samples ({len(test_df)/n:.1%})")
    print(f"\nChurn rates:")
    print(f"  Train: {train_df['churn_label'].mean():.2%}")
    print(f"  Val:   {val_df['churn_label'].mean():.2%}")
    print(f"  Test:  {test_df['churn_label'].mean():.2%}")
    
    return train_df, val_df, test_df

train_df, val_df, test_df = split_data(sequence_df, train_ratio=0.7, val_ratio=0.15)

In [ ]:
# Detailed Class Distribution Analysis
print("\n" + "="*70)
print("CLASS DISTRIBUTION ANALYSIS")
print("="*70)

print("\n📊 Overall Dataset:")
print(f"  Total accounts: {len(sequence_df):,}")
print(f"  Churned accounts: {sequence_df['churn_label'].sum():,} ({sequence_df['churn_label'].mean():.2%})")
print(f"  Active accounts: {(len(sequence_df) - sequence_df['churn_label'].sum()):,} ({(1-sequence_df['churn_label'].mean()):.2%})")

print("\n📊 Train/Val/Test Split:")
print(f"  Train: {len(train_df):,} samples")
print(f"    - Churned: {train_df['churn_label'].sum():,} ({train_df['churn_label'].mean():.2%})")
print(f"    - Active: {(len(train_df) - train_df['churn_label'].sum()):,} ({(1-train_df['churn_label'].mean()):.2%})")

print(f"\n  Val: {len(val_df):,} samples")
print(f"    - Churned: {val_df['churn_label'].sum():,} ({val_df['churn_label'].mean():.2%})")
print(f"    - Active: {(len(val_df) - val_df['churn_label'].sum()):,} ({(1-val_df['churn_label'].mean()):.2%})")

print(f"\n  Test: {len(test_df):,} samples")
print(f"    - Churned: {test_df['churn_label'].sum():,} ({test_df['churn_label'].mean():.2%})")
print(f"    - Active: {(len(test_df) - test_df['churn_label'].sum()):,} ({(1-test_df['churn_label'].mean()):.2%})")

# Calculate class imbalance ratio
churn_ratio = sequence_df['churn_label'].mean()
imbalance_ratio = (1 - churn_ratio) / churn_ratio if churn_ratio > 0 else float('inf')

print(f"\n⚖️ Class Imbalance:")
print(f"  Imbalance ratio: {imbalance_ratio:.1f}:1 (Active:Churn)")

if imbalance_ratio > 10:
    print(f"  ⚠️ WARNING: Severe class imbalance detected!")
    print(f"  Recommendations:")
    print(f"    1. Use class weights in loss function")
    print(f"    2. Consider SMOTE or other resampling techniques")
    print(f"    3. Adjust classification threshold (lower than 0.5)")
    print(f"    4. Use focal loss instead of BCE")
elif imbalance_ratio > 3:
    print(f"  ⚠️ Moderate class imbalance - consider using class weights")
else:
    print(f"  ✓ Class distribution is relatively balanced")

## 6. Define PyTorch Dataset

In [ ]:
# 🔍 DATA LEAKAGE CHECK
print("\n" + "="*70)
print("DATA LEAKAGE DIAGNOSTICS")
print("="*70)

# Check if there's any overlap between train and test accounts
train_accounts = set(train_df['account_id'])
test_accounts = set(test_df['account_id'])
overlap = train_accounts & test_accounts

print(f"\nAccount Overlap Check:")
print(f"  Train accounts: {len(train_accounts):,}")
print(f"  Test accounts: {len(test_accounts):,}")
print(f"  Overlap: {len(overlap):,}")

if len(overlap) > 0:
    print(f"  ⚠️ WARNING: Train and test sets have overlapping accounts!")
    print(f"  This causes data leakage!")
else:
    print(f"  ✓ No overlap - good!")

In [ ]:
class ChurnDataset(Dataset):
    """Custom Dataset for churn prediction with time series data"""
    
    def __init__(self, df, max_lookback_window=12):
        self.df = df
        self.max_lookback_window = max_lookback_window
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        # Get sequence and label
        sequence = self.df.iloc[idx]['sequence']
        label = self.df.iloc[idx]['churn_label']
        
        # Convert to tensors
        sequence = torch.tensor(sequence, dtype=torch.float32)
        label = torch.tensor(label, dtype=torch.float32)
        
        # Pad or truncate sequence to max_lookback_window
        seq_len = sequence.shape[0]
        if seq_len < self.max_lookback_window:
            # Pad with zeros at the beginning
            padding = torch.zeros((self.max_lookback_window - seq_len, sequence.shape[1]))
            sequence = torch.cat([padding, sequence], dim=0)
        elif seq_len > self.max_lookback_window:
            # Take last max_lookback_window timesteps
            sequence = sequence[-self.max_lookback_window:, :]
        
        return sequence, label

# Configuration
MAX_LOOKBACK_WINDOW = 12
BATCH_SIZE = 32
N_FEATURES = 10  # Number of features per timestep

# Create datasets
train_dataset = ChurnDataset(train_df, MAX_LOOKBACK_WINDOW)
val_dataset = ChurnDataset(val_df, MAX_LOOKBACK_WINDOW)
test_dataset = ChurnDataset(test_df, MAX_LOOKBACK_WINDOW)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("✓ DataLoaders created")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

# Test data loading
sample_batch = next(iter(train_loader))
print(f"\nSample batch shapes:")
print(f"  Sequences: {sample_batch[0].shape}")  # [batch_size, seq_len, n_features]
print(f"  Labels: {sample_batch[1].shape}")     # [batch_size]

## 7. Define LSTM with Attention Model

In [ ]:
class LSTMWithAttention(nn.Module):
    """LSTM model with attention mechanism for sequence classification"""
    
    def __init__(self, input_size, hidden_size, output_size, num_layers=2, dropout=0.2):
        super(LSTMWithAttention, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # LSTM layer
        self.lstm = nn.LSTM(
            input_size, 
            hidden_size, 
            num_layers=num_layers, 
            batch_first=True, 
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Attention layer
        self.attention = nn.Linear(hidden_size, 1)
        
        # Fully connected layers
        self.fc1 = nn.Linear(hidden_size, hidden_size // 2)
        self.fc2 = nn.Linear(hidden_size // 2, output_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x, return_attention=False):
        # LSTM output: [batch, seq_len, hidden_size]
        lstm_out, (hidden, cell) = self.lstm(x)
        
        # Attention mechanism
        # attention_weights: [batch, seq_len, 1]
        attention_weights = torch.softmax(self.attention(lstm_out), dim=1)
        
        # context_vector: [batch, hidden_size]
        context_vector = torch.sum(attention_weights * lstm_out, dim=1)
        
        # Fully connected layers
        out = self.relu(self.fc1(context_vector))
        out = self.dropout(out)
        out = self.fc2(out)
        out = self.sigmoid(out)
        
        if return_attention:
            return out, attention_weights
        return out

# Initialize model
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.3

model = LSTMWithAttention(
    input_size=N_FEATURES,
    hidden_size=HIDDEN_SIZE,
    output_size=1,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(device)

print("✓ Model initialized")
print(f"\nModel architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## 8. Define Training and Evaluation Functions

In [ ]:
def train_epoch(model, device, train_loader, criterion, optimizer):
    """Train model for one epoch"""
    model.train()
    running_loss = 0.0
    
    for sequence, labels in tqdm(train_loader, desc="Training", leave=False):
        sequence, labels = sequence.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(sequence)
        labels = labels.reshape(-1, 1)
        loss = criterion(outputs, labels)
        
        # Handle both scalar and tensor losses
        if loss.dim() > 0:
            loss = loss.mean()
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * labels.size(0)
    
    return running_loss / len(train_loader.dataset)


def evaluate_model(model, device, data_loader, criterion, threshold=0.5):
    """Evaluate model on a dataset"""
    model.eval()
    running_loss = 0.0
    y_probs = []
    y_labels = []
    
    with torch.no_grad():
        for sequence, labels in data_loader:
            sequence, labels = sequence.to(device), labels.to(device)
            outputs = model(sequence)
            labels = labels.reshape(-1, 1)
            loss = criterion(outputs, labels)
            
            # Handle both scalar and tensor losses (for reduction='none')
            if loss.dim() > 0:
                loss = loss.mean()
            
            running_loss += loss.item() * labels.size(0)
            
            # Save predictions and labels
            y_probs.append(outputs.cpu())
            y_labels.append(labels.cpu())
    
    # Concatenate all batches
    y_probs = torch.cat(y_probs).numpy().flatten()
    y_labels = torch.cat(y_labels).numpy().flatten()
    
    # Calculate metrics
    avg_loss = running_loss / len(data_loader.dataset)
    y_pred = (y_probs > threshold).astype(int)
    
    precision = precision_score(y_labels, y_pred, zero_division=0)
    recall = recall_score(y_labels, y_pred, zero_division=0)
    f1 = f1_score(y_labels, y_pred, zero_division=0)
    
    return avg_loss, precision, recall, f1, y_probs, y_labels

print("✓ Training and evaluation functions defined")

## 9. Train the Model

In [ ]:
# Training configuration
LEARNING_RATE = 0.001
NUM_EPOCHS = 50
PATIENCE = 7
THRESHOLD = 0.5

# Calculate class weights to handle imbalance
num_no_churn = len(train_df) - train_df['churn_label'].sum()
num_churn = train_df['churn_label'].sum()
total = len(train_df)

print(f"\n⚖️ Class Distribution in Training Data:")
print(f"  No Churn samples: {num_no_churn:,}")
print(f"  Churn samples: {num_churn:,}")
print(f"  Total samples: {total:,}")

# Check if data is valid for training
if num_churn == 0 or num_no_churn == 0:
    print("\n" + "="*70)
    print("❌ CRITICAL ERROR: Cannot train with only one class!")
    print("="*70)
    if num_churn == 0:
        print("  Issue: No churned accounts found in training data")
    else:
        print("  Issue: No active (non-churned) accounts found in training data")
    print("\n📋 Recommended Actions:")
    print("  1. Check your CHURN_RECORDS table - it might be marking ALL accounts as churned")
    print("  2. Verify the churn labeling logic in create_churn_sequences()")
    print("  3. Check if CHURN_RECORDS.USERID contains all account IDs from PHONE_USAGE_DATA")
    print("  4. Review your data generation process")
    print("\n⚠️ Training cannot proceed. Please fix the data issue first.")
    raise ValueError("Training data contains only one class. Need both churned and non-churned accounts.")

# Weight for positive class (churn) - higher weight for minority class
pos_weight = torch.tensor([num_no_churn / num_churn]).to(device)
print(f"  Positive class weight (for churn): {pos_weight.item():.2f}")

# Initialize optimizer and loss function with class weights
criterion = nn.BCELoss(reduction='none')  # We'll apply weights manually
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Training loop with early stopping and class weights
best_val_loss = float('inf')
best_f1 = 0
patience_counter = 0
best_model_state = None  # Initialize to avoid NameError
history = {
    'train_loss': [], 'val_loss': [],
    'val_precision': [], 'val_recall': [], 'val_f1': []
}

print("\n" + "="*70)
print("Starting Training (with Class Weighting)")
print("="*70)

for epoch in range(NUM_EPOCHS):
    # Train with weighted loss
    model.train()
    running_loss = 0.0
    
    for sequence, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", leave=False):
        sequence, labels = sequence.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(sequence)
        labels = labels.reshape(-1, 1)
        
        # Calculate loss with class weights
        loss_per_sample = criterion(outputs, labels)
        # Apply higher weight to churn samples (label=1)
        weights = torch.where(labels == 1, pos_weight.squeeze(), torch.tensor(1.0).to(device))
        loss = (loss_per_sample * weights).mean()
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * labels.size(0)
    
    train_loss = running_loss / len(train_loader.dataset)
    
    # Validate (without class weights)
    val_loss, precision, recall, f1, _, _ = evaluate_model(
        model, device, val_loader, nn.BCELoss(), THRESHOLD
    )
    
    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_precision'].append(precision)
    history['val_recall'].append(recall)
    history['val_f1'].append(f1)
    
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print(f"  Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print(f"  Val Metrics - P: {precision:.4f} | R: {recall:.4f} | F1: {f1:.4f}")
    
    # Early stopping based on F1 score
    if f1 > best_f1:
        best_f1 = f1
        best_val_loss = val_loss
        patience_counter = 0
        # Save best model
        best_model_state = model.state_dict()
        print("  ✓ New best model!")
    else:
        patience_counter += 1
        print(f"  Patience: {patience_counter}/{PATIENCE}")
    
    if patience_counter >= PATIENCE:
        print(f"\n✓ Early stopping triggered after {epoch+1} epochs")
        break

# Load best model (if one was saved)
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print("\n" + "="*70)
    print("Training completed! Best model loaded.")
    print(f"Best F1 Score: {best_f1:.4f}")
    print("="*70)
else:
    print("\n⚠️ Warning: No improvement during training. Using final model state.")

## 10. Evaluate on Test Set

In [ ]:
print("\n" + "="*70)
print("Final Evaluation on Test Set")
print("="*70)

test_loss, test_precision, test_recall, test_f1, test_probs, test_labels = evaluate_model(
    model, device, test_loader, criterion, THRESHOLD
)

# Debug: Check prediction distribution
print(f"\n🔍 Prediction Distribution Debug:")
print(f"  Test probabilities - Min: {test_probs.min():.6f}, Max: {test_probs.max():.6f}, Mean: {test_probs.mean():.6f}")
print(f"  Unique test labels: {np.unique(test_labels)}")
test_pred = (test_probs > THRESHOLD).astype(int)
print(f"  Unique predictions: {np.unique(test_pred)}")
print(f"  Predicted as No Churn (0): {np.sum(test_pred == 0)} ({np.sum(test_pred == 0)/len(test_pred)*100:.1f}%)")
print(f"  Predicted as Churn (1): {np.sum(test_pred == 1)} ({np.sum(test_pred == 1)/len(test_pred)*100:.1f}%)")
print(f"  Actual No Churn (0): {np.sum(test_labels == 0)} ({np.sum(test_labels == 0)/len(test_labels)*100:.1f}%)")
print(f"  Actual Churn (1): {np.sum(test_labels == 1)} ({np.sum(test_labels == 1)/len(test_labels)*100:.1f}%)")

# Calculate AUC (handle case where only one class is present)
try:
    if len(np.unique(test_labels)) > 1:
        fpr, tpr, _ = roc_curve(test_labels, test_probs)
        test_auc = auc(fpr, tpr)
    else:
        print("  ⚠️ Warning: Only one class present in test labels, cannot calculate AUC")
        test_auc = float('nan')
        fpr, tpr = None, None
except Exception as e:
    print(f"  ⚠️ Warning: Error calculating AUC: {e}")
    test_auc = float('nan')
    fpr, tpr = None, None

# Calculate confusion matrix with labels parameter to ensure 2x2 matrix
cm = confusion_matrix(test_labels, test_pred, labels=[0, 1])

print(f"\nTest Set Results:")
print(f"  Loss:      {test_loss:.4f}")
print(f"  Precision: {test_precision:.4f}")
print(f"  Recall:    {test_recall:.4f}")
print(f"  F1 Score:  {test_f1:.4f}")
print(f"  AUC-ROC:   {test_auc:.4f}")

# Print confusion matrix safely
print(f"\nConfusion Matrix:")
if cm.shape == (2, 2):
    print(f"                Predicted")
    print(f"              No Churn  Churn")
    print(f"Actual No Churn  {cm[0,0]:4d}    {cm[0,1]:4d}")
    print(f"       Churn     {cm[1,0]:4d}    {cm[1,1]:4d}")
else:
    print(f"  ⚠️ Warning: Unexpected confusion matrix shape: {cm.shape}")
    print(cm)

# Classification report
print(f"\nDetailed Classification Report:")
try:
    print(classification_report(test_labels, test_pred, target_names=['No Churn', 'Churn'], zero_division=0))
except Exception as e:
    print(f"  ⚠️ Error generating classification report: {e}")

# Additional diagnostic: Show sample predictions
print(f"\n📊 Sample Predictions (first 20):")
sample_df = pd.DataFrame({
    'Actual': test_labels[:20],
    'Probability': test_probs[:20],
    'Predicted': test_pred[:20]
})
print(sample_df.to_string(index=False))

## 11. Visualizations

In [ ]:
# Training History - Professional styling
fig, axes = plt.subplots(2, 2, figsize=(10, 7))  # Smaller: was 15x10

# Loss
axes[0, 0].plot(history['train_loss'], label='Train', linewidth=1.5)
axes[0, 0].plot(history['val_loss'], label='Validation', linewidth=1.5)
axes[0, 0].set_xlabel('Epoch', fontsize=9)
axes[0, 0].set_ylabel('Loss', fontsize=9)
axes[0, 0].set_title('Training and Validation Loss', fontsize=10, fontweight='bold')
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid(True, alpha=0.3, linewidth=0.5)
axes[0, 0].tick_params(labelsize=8)

# Precision
axes[0, 1].plot(history['val_precision'], color='#2E86AB', linewidth=1.5)
axes[0, 1].set_xlabel('Epoch', fontsize=9)
axes[0, 1].set_ylabel('Precision', fontsize=9)
axes[0, 1].set_title('Validation Precision', fontsize=10, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3, linewidth=0.5)
axes[0, 1].set_ylim([0, 1])
axes[0, 1].tick_params(labelsize=8)

# Recall
axes[1, 0].plot(history['val_recall'], color='#A23B72', linewidth=1.5)
axes[1, 0].set_xlabel('Epoch', fontsize=9)
axes[1, 0].set_ylabel('Recall', fontsize=9)
axes[1, 0].set_title('Validation Recall', fontsize=10, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, linewidth=0.5)
axes[1, 0].set_ylim([0, 1])
axes[1, 0].tick_params(labelsize=8)

# F1 Score
axes[1, 1].plot(history['val_f1'], color='#F18F01', linewidth=1.5)
axes[1, 1].set_xlabel('Epoch', fontsize=9)
axes[1, 1].set_ylabel('F1 Score', fontsize=9)
axes[1, 1].set_title('Validation F1 Score', fontsize=10, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, linewidth=0.5)
axes[1, 1].set_ylim([0, 1])
axes[1, 1].tick_params(labelsize=8)

plt.tight_layout()
plt.show()

print("✓ Training history plotted")

In [ ]:
# ROC Curve and Confusion Matrix - Professional styling
fig, axes = plt.subplots(1, 2, figsize=(10, 4))  # Smaller: was 15x5

# ROC Curve
if fpr is not None and tpr is not None and not np.isnan(test_auc):
    axes[0].plot(fpr, tpr, label=f'AUC = {test_auc:.3f}', linewidth=2, color='#2E86AB')
    axes[0].plot([0, 1], [0, 1], 'k--', label='Random', linewidth=1, alpha=0.5)
    axes[0].set_xlabel('False Positive Rate', fontsize=9)
    axes[0].set_ylabel('True Positive Rate', fontsize=9)
    axes[0].set_title('ROC Curve', fontsize=10, fontweight='bold')
    axes[0].legend(fontsize=8, loc='lower right')
    axes[0].grid(True, alpha=0.3, linewidth=0.5)
    axes[0].tick_params(labelsize=8)
else:
    axes[0].text(0.5, 0.5, 'ROC Curve\nnot available', 
                ha='center', va='center', fontsize=9, transform=axes[0].transAxes)
    axes[0].set_xlabel('False Positive Rate', fontsize=9)
    axes[0].set_ylabel('True Positive Rate', fontsize=9)
    axes[0].set_title('ROC Curve', fontsize=10, fontweight='bold')
    axes[0].tick_params(labelsize=8)

# Confusion Matrix
if cm.shape == (2, 2):
    im = axes[1].imshow(cm, interpolation='nearest', cmap='Blues')
    cbar = axes[1].figure.colorbar(im, ax=axes[1])
    cbar.ax.tick_params(labelsize=7)
    axes[1].set(xticks=[0, 1], yticks=[0, 1],
                xticklabels=['No Churn', 'Churn'],
                yticklabels=['No Churn', 'Churn'])
    axes[1].set_xlabel('Predicted', fontsize=9)
    axes[1].set_ylabel('Actual', fontsize=9)
    axes[1].set_title('Confusion Matrix', fontsize=10, fontweight='bold')
    axes[1].tick_params(labelsize=8)
    
    # Add text annotations with smaller font
    thresh = cm.max() / 2
    for i in range(2):
        for j in range(2):
            text = axes[1].text(j, i, f'{cm[i, j]}\n({cm[i, j]/cm.sum()*100:.1f}%)',
                               ha="center", va="center",
                               color="white" if cm[i, j] > thresh else "black",
                               fontsize=9, fontweight='bold')
else:
    axes[1].text(0.5, 0.5, f'Confusion Matrix\nShape: {cm.shape}', 
                ha='center', va='center', fontsize=9, transform=axes[1].transAxes)
    axes[1].set_title('Confusion Matrix', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ ROC curve and confusion matrix plotted")

In [ ]:
# Precision-Recall Curve - Professional styling
try:
    if len(np.unique(test_labels)) > 1:
        precision_vals, recall_vals, thresholds_pr = precision_recall_curve(test_labels, test_probs)
        
        fig, ax = plt.subplots(figsize=(6, 4))  # Smaller: was 8x6
        ax.plot(recall_vals, precision_vals, linewidth=2, color='#A23B72')
        ax.set_xlabel('Recall', fontsize=9)
        ax.set_ylabel('Precision', fontsize=9)
        ax.set_title('Precision-Recall Curve', fontsize=10, fontweight='bold')
        ax.grid(True, alpha=0.3, linewidth=0.5)
        ax.tick_params(labelsize=8)
        plt.tight_layout()
        plt.show()
        
        print("✓ Precision-Recall curve plotted")
    else:
        print("⚠️ Precision-Recall curve not available (only one class in labels)")
except Exception as e:
    print(f"⚠️ Error plotting Precision-Recall curve: {e}")

## 12. Generate Predictions for All Accounts

In [ ]:
# Generate predictions for test set
model.eval()
test_predictions = []

with torch.no_grad():
    for idx in range(len(test_df)):
        sequence, label = test_dataset[idx]
        sequence = sequence.unsqueeze(0).to(device)  # Add batch dimension
        
        # Get prediction and attention weights
        prob, attention = model(sequence, return_attention=True)
        
        test_predictions.append({
            'account_id': test_df.iloc[idx]['account_id'],
            'actual_churn': int(label.item()),
            'churn_probability': float(prob.cpu().item()),
            'predicted_churn': int((prob.cpu().item() > THRESHOLD)),
            'sequence_length': int(test_df.iloc[idx]['seq_length'])
        })

# Create predictions DataFrame
predictions_df = pd.DataFrame(test_predictions)

print(f"\n✓ Generated predictions for {len(predictions_df):,} test accounts")
print(f"\nSample predictions:")
print(predictions_df.head(10))

# Show high-risk accounts
print(f"\n=== High Churn Risk Accounts (Probability > 0.7) ===")
high_risk = predictions_df[predictions_df['churn_probability'] > 0.7].sort_values('churn_probability', ascending=False)
print(high_risk[['account_id', 'churn_probability', 'actual_churn', 'predicted_churn']].head(10))

## 13. Save Results to Snowflake

In [ ]:
# Save predictions to Snowflake (MY_DATABASE.PUBLIC)
print("\nSaving predictions to MY_DATABASE.PUBLIC.CHURN_PREDICTIONS...")

try:
    # Create Snowpark DataFrame from predictions
    predictions_snowpark = session.create_dataframe(predictions_df)
    
    # Write to table in MY_DATABASE.PUBLIC
    predictions_snowpark.write.mode("overwrite").save_as_table("MY_DATABASE.PUBLIC.CHURN_PREDICTIONS")
    
    print(f"✓ Predictions saved to MY_DATABASE.PUBLIC.CHURN_PREDICTIONS table")
    print(f"  Rows: {len(predictions_df):,}")
    
    # Verify
    result_count = session.table("MY_DATABASE.PUBLIC.CHURN_PREDICTIONS").count()
    print(f"  Verified row count: {result_count:,}")
    
except Exception as e:
    print(f"✗ Error saving predictions: {str(e)}")

In [ ]:
# Save model metrics to Snowflake (MY_DATABASE.PUBLIC)
print("\nSaving model metrics to MY_DATABASE.PUBLIC.CHURN_MODEL_METRICS...")

try:
    # Create metrics DataFrame
    metrics_df = pd.DataFrame({
        'model_name': ['LSTM_with_Attention'],
        'train_date': [datetime.now()],
        'test_loss': [test_loss],
        'test_precision': [test_precision],
        'test_recall': [test_recall],
        'test_f1_score': [test_f1],
        'test_auc_roc': [test_auc],
        'num_features': [N_FEATURES],
        'lookback_window': [MAX_LOOKBACK_WINDOW],
        'hidden_size': [HIDDEN_SIZE],
        'num_layers': [NUM_LAYERS],
        'learning_rate': [LEARNING_RATE],
        'train_samples': [len(train_df)],
        'test_samples': [len(test_df)]
    })
    
    # Save to Snowflake in MY_DATABASE.PUBLIC
    metrics_snowpark = session.create_dataframe(metrics_df)
    metrics_snowpark.write.mode("append").save_as_table("MY_DATABASE.PUBLIC.CHURN_MODEL_METRICS")
    
    print(f"✓ Model metrics saved to MY_DATABASE.PUBLIC.CHURN_MODEL_METRICS table")
    
except Exception as e:
    print(f"✗ Error saving metrics: {str(e)}")

## 14. Summary and Next Steps

In [ ]:
# ============================================================================
# LOAD MODEL - Example code (uncomment to use)
# ============================================================================

# IMPORTANT: You must define the LSTMWithAttention class before loading
# (It's already defined in cell 19 of this notebook)

# Load from PyTorch checkpoint (.pt format - RECOMMENDED for PyTorch models)
# Uncomment the following lines to load:

"""
# Load the checkpoint using torch.load()
# map_location ensures it works on both CPU and GPU machines
checkpoint = torch.load('churn_model_v2.pt', map_location=device)

# Create a new model instance using the saved configuration
loaded_model = LSTMWithAttention(
    input_size=checkpoint['model_config']['input_size'],
    hidden_size=checkpoint['model_config']['hidden_size'],
    output_size=checkpoint['model_config']['output_size'],
    num_layers=checkpoint['model_config']['num_layers'],
    dropout=checkpoint['model_config']['dropout']
).to(device)

# Load the saved weights into the model
loaded_model.load_state_dict(checkpoint['model_state_dict'])
loaded_model.eval()

print("✓ Model loaded successfully!")
print(f"\n📊 Model Information:")
print(f"  Trained on: {checkpoint['metadata']['train_date']}")
print(f"  Test F1: {checkpoint['test_metrics']['f1']:.4f}")
print(f"  Test Precision: {checkpoint['test_metrics']['precision']:.4f}")
print(f"  Test Recall: {checkpoint['test_metrics']['recall']:.4f}")
print(f"  Test AUC: {checkpoint['test_metrics']['auc']:.4f}")
print(f"\n🔧 Configuration:")
print(f"  Features: {checkpoint['metadata']['num_features']}")
print(f"  Lookback window: {checkpoint['metadata']['lookback_window']} months")
print(f"  Hidden size: {checkpoint['model_config']['hidden_size']}")
print(f"  Layers: {checkpoint['model_config']['num_layers']}")
print(f"  Threshold: {checkpoint['threshold']:.2f}")
"""

# Example: Make predictions with loaded model
# Uncomment the following to test:

"""
# Get a sample from test set
sample_sequence, sample_label = test_dataset[0]
sample_sequence = sample_sequence.unsqueeze(0).to(device)

# Make prediction
with torch.no_grad():
    prediction = loaded_model(sample_sequence)
    prob = prediction.item()
    predicted_class = 'Churn' if prob > checkpoint['threshold'] else 'No Churn'
    actual_class = 'Churn' if sample_label.item() == 1 else 'No Churn'
    
    print(f"\n🔮 Sample Prediction:")
    print(f"  Churn probability: {prob:.4f}")
    print(f"  Predicted: {predicted_class}")
    print(f"  Actual: {actual_class}")
    print(f"  Correct: {'✓' if predicted_class == actual_class else '✗'}")
"""

print("\n💡 Uncomment the code above to load and test the model.")
print("   Using torch.load() with .pt files is the recommended approach for PyTorch models!")

In [ ]:
# ============================================================================
# LOAD MODEL - Alternative Methods
# ============================================================================

# Method 1: Load state_dict only (lightweight, RECOMMENDED)
# Best for: Production inference, model deployment
# Uncomment the following lines to load:

"""
# Load checkpoint
checkpoint = torch.load('churn_model_v2.pt', map_location=device)

# Reconstruct model from config
loaded_model = LSTMWithAttention(
    input_size=checkpoint['model_config']['input_size'],
    hidden_size=checkpoint['model_config']['hidden_size'],
    output_size=checkpoint['model_config']['output_size'],
    num_layers=checkpoint['model_config']['num_layers'],
    dropout=checkpoint['model_config']['dropout']
).to(device)

# Load weights
loaded_model.load_state_dict(checkpoint['model_state_dict'])
loaded_model.eval()

print("✓ Model loaded from state_dict")
print(f"  Trained on: {checkpoint['metadata']['train_date']}")
print(f"  Test F1: {checkpoint['test_metrics']['f1']:.4f}")
print(f"  Test AUC: {checkpoint['test_metrics']['auc']:.4f}")
"""

# Method 2: Save/Load entire model (simpler but less flexible)
# Note: Requires the model class to be importable when loading
# Uncomment the following to save/load complete model:

"""
# To save entire model (alternative approach):
# torch.save(model, 'churn_model_complete.pt')

# To load entire model:
loaded_model = torch.load('churn_model_complete.pt', map_location=device)
loaded_model.eval()
print("✓ Complete model loaded")
"""

# Example: Make a prediction with loaded model
"""
# Prepare sample data
sample_sequence, sample_label = test_dataset[0]
sample_sequence = sample_sequence.unsqueeze(0).to(device)

# Make prediction
with torch.no_grad():
    prediction = loaded_model(sample_sequence)
    threshold = checkpoint['threshold']
    print(f"\nSample prediction:")
    print(f"  Churn probability: {prediction.item():.4f}")
    print(f"  Predicted class: {'Churn' if prediction.item() > threshold else 'No Churn'}")
    print(f"  Actual class: {'Churn' if sample_label.item() == 1 else 'No Churn'}")
"""

print("\n💡 PyTorch Model Saving Best Practices:")
print("   1. Use torch.save(checkpoint_dict, 'model.pt') - saves state_dict + metadata")
print("   2. Use .pt or .pth extension for PyTorch models")
print("   3. Include model config in checkpoint for easy reconstruction")
print("   4. Use map_location when loading to handle CPU/GPU differences")

## 16. Load Model (Example Code)

In [ ]:
# ============================================================================
# SAVE MODEL - VERSION 2 (Using PyTorch native format)
# ============================================================================

print("\n" + "="*70)
print("SAVING MODEL - VERSION 2 (PyTorch .pt format)")
print("="*70)

# Create model checkpoint with complete metadata
model_checkpoint = {
    'version': 'v2',
    'model_state_dict': model.state_dict(),
    'model_config': {
        'input_size': N_FEATURES,
        'hidden_size': HIDDEN_SIZE,
        'output_size': 1,
        'num_layers': NUM_LAYERS,
        'dropout': DROPOUT
    },
    'threshold': THRESHOLD,
    'test_metrics': {
        'f1': test_f1,
        'precision': test_precision,
        'recall': test_recall,
        'auc': test_auc,
        'loss': test_loss
    },
    'metadata': {
        'version': 'v2',
        'description': 'Improved model with enhanced churn rate handling and pattern recognition',
        'train_date': datetime.now().isoformat(),
        'num_features': N_FEATURES,
        'lookback_window': MAX_LOOKBACK_WINDOW,
        'hidden_size': HIDDEN_SIZE,
        'num_layers': NUM_LAYERS,
        'dropout': DROPOUT,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'num_epochs_trained': len(history['train_loss']),
        'train_samples': len(train_df),
        'test_samples': len(test_df),
        'changes_from_v1': [
            'Enhanced churn pattern recognition',
            'Improved class imbalance handling',
            'Optimized feature normalization',
            'Better threshold calibration'
        ]
    }
}

# Save using PyTorch native format (.pt) - RECOMMENDED for PyTorch models
# torch.save() uses pickle internally but is optimized for PyTorch tensors
torch.save(model_checkpoint, 'churn_model_v2.pt')
print("✓ Saved: churn_model_v2.pt (PyTorch native format)")

# Upload to Snowflake stage
print("\n📤 Uploading to Snowflake...")
try:
    # Create versioned directory in stage
    session.sql("CREATE STAGE IF NOT EXISTS MY_DATABASE.PUBLIC.MODELS").collect()
    
    # Upload .pt file to v2 directory
    session.file.put(
        'churn_model_v2.pt',
        '@MY_DATABASE.PUBLIC.MODELS/v2/',
        auto_compress=False,
        overwrite=True
    )
    print("✓ Uploaded to: @MY_DATABASE.PUBLIC.MODELS/v2/churn_model_v2.pt")
    
    # Register in MODEL_REGISTRY
    registry_df = pd.DataFrame({
        'VERSION': ['v2'],
        'MODEL_PATH': ['@MY_DATABASE.PUBLIC.MODELS/v2/churn_model_v2.pt'],
        'DESCRIPTION': ['Improved model with enhanced churn rate handling and pattern recognition'],
        'F1_SCORE': [test_f1],
        'PRECISION': [test_precision],
        'RECALL': [test_recall],
        'AUC': [test_auc],
        'TRAIN_DATE': [datetime.now()],
        'IS_PRODUCTION': [False]
    })
    
    # Create table if not exists
    session.sql("""
        CREATE TABLE IF NOT EXISTS MY_DATABASE.PUBLIC.MODEL_REGISTRY (
            VERSION VARCHAR(50) PRIMARY KEY,
            MODEL_PATH VARCHAR(500),
            DESCRIPTION VARCHAR(1000),
            F1_SCORE FLOAT,
            PRECISION FLOAT,
            RECALL FLOAT,
            AUC FLOAT,
            TRAIN_DATE TIMESTAMP,
            IS_PRODUCTION BOOLEAN DEFAULT FALSE
        )
    """).collect()
    
    # Insert or update
    session.sql("""
        MERGE INTO MY_DATABASE.PUBLIC.MODEL_REGISTRY AS target
        USING (SELECT 'v2' AS VERSION) AS source
        ON target.VERSION = source.VERSION
        WHEN MATCHED THEN UPDATE SET
            MODEL_PATH = '@MY_DATABASE.PUBLIC.MODELS/v2/churn_model_v2.pt',
            DESCRIPTION = 'Improved model with enhanced churn rate handling and pattern recognition',
            F1_SCORE = {},
            PRECISION = {},
            RECALL = {},
            AUC = {},
            TRAIN_DATE = CURRENT_TIMESTAMP()
        WHEN NOT MATCHED THEN INSERT (VERSION, MODEL_PATH, DESCRIPTION, F1_SCORE, PRECISION, RECALL, AUC, TRAIN_DATE)
        VALUES ('v2', '@MY_DATABASE.PUBLIC.MODELS/v2/churn_model_v2.pt', 
                'Improved model with enhanced churn rate handling and pattern recognition',
                {}, {}, {}, {}, CURRENT_TIMESTAMP())
    """.format(test_f1, test_precision, test_recall, test_auc,
               test_f1, test_precision, test_recall, test_auc)).collect()
    
    print("✓ Registered in MODEL_REGISTRY")
    
except Exception as e:
    print(f"⚠️ Error uploading to Snowflake: {e}")

print("\n" + "="*70)
print("✓ MODEL v2 SAVED SUCCESSFULLY")
print("="*70)
print(f"\n📊 Version: v2")
print(f"📁 Local file: churn_model_v2.pt (PyTorch native format)")
print(f"☁️  Snowflake: @MY_DATABASE.PUBLIC.MODELS/v2/")
print(f"\n📈 Performance:")
print(f"  F1: {test_f1:.4f}")
print(f"  Precision: {test_precision:.4f}")
print(f"  Recall: {test_recall:.4f}")
print(f"  AUC: {test_auc:.4f}")
print(f"\n💡 Note: Using torch.save() with .pt extension is the recommended")
print(f"   format for PyTorch models. It handles tensors efficiently and")
print(f"   preserves GPU/CPU device information.")

In [ ]:
! ls

In [ ]:
# ============================================================================
# UPLOAD MODEL FILES TO SNOWFLAKE STAGE
# ============================================================================

print("\n" + "="*70)
print("UPLOADING MODEL FILES TO SNOWFLAKE STAGE")
print("="*70)

# Create a stage if it doesn't exist
try:
    session.sql("CREATE STAGE IF NOT EXISTS MY_DATABASE.PUBLIC.MODELS").collect()
    print("✓ Stage MY_DATABASE.PUBLIC.MODELS ready")
except Exception as e:
    print(f"⚠️ Error creating stage: {e}")

# Upload the model file to the stage (PyTorch .pt format)
try:
    # Upload PyTorch file (.pt is the recommended format for PyTorch models)
    session.file.put(
        'churn_model_v2.pt',
        '@MY_DATABASE.PUBLIC.MODELS',
        auto_compress=False,
        overwrite=True
    )
    print("✓ Uploaded: churn_model_v2.pt")

    print(f"\n📦 Model file uploaded to: @MY_DATABASE.PUBLIC.MODELS")
    print(f"\n💡 To list files in the stage, run:")
    print(f"   LIST @MY_DATABASE.PUBLIC.MODELS")

except Exception as e:
    print(f"✗ Error uploading files: {e}")

# List files in the stage to verify
print("\n" + "="*70)
print("FILES IN STAGE")
print("="*70)
try:
    files = session.sql("LIST @MY_DATABASE.PUBLIC.MODELS").collect()
    for file in files:
        print(f"  - {file['name']}")
except Exception as e:
    print(f"Error listing files: {e}")

In [ ]:
# -- To download the model file later:

# Download model from Snowflake stage to local
session.file.get(
    '@MY_DATABASE.PUBLIC.MODELS/churn_model_v2.pt',
    '/tmp/'  # or any local directory
)

print("✓ Model downloaded to /tmp/churn_model_v2.pt")

## 15. Save Model to File

In [ ]:
import torch
from snowflake.snowpark.context import get_active_session

# Get active session
session = get_active_session()

# ============================================================================
# DOWNLOAD MODEL FROM SNOWFLAKE STAGE
# ============================================================================

print("Downloading model from Snowflake stage...")

# Download the .pt file from stage to notebook's local filesystem
session.file.get(
    '@MY_DATABASE.PUBLIC.MODELS/churn_model_v2.pt',  # Stage path
    '/tmp/'  # Local directory in notebook
)

print("✓ Model downloaded to /tmp/churn_model_v2.pt")

In [ ]:
# ============================================================================
# LOAD THE MODEL (PyTorch native format)
# ============================================================================

# Load the PyTorch checkpoint file
# torch.load() is the recommended way to load PyTorch models
model_data = torch.load('/tmp/churn_model_v2.pt', map_location=torch.device('cpu'))

print("✓ Model checkpoint loaded successfully!")
print(f"\n📋 Checkpoint contents:")
print(f"  - Version: {model_data.get('version', 'N/A')}")
print(f"  - Model config: {model_data['model_config']}")
print(f"  - Threshold: {model_data['threshold']}")
print(f"  - Test metrics: {model_data['test_metrics']}")

In [ ]:
model

In [ ]:
import torch
import numpy as np

# ============================================================================
# STEP 1: LOAD MODEL WEIGHTS INTO MODEL INSTANCE
# ============================================================================

print("\n" + "="*70)
print("LOADING MODEL FOR PREDICTIONS")
print("="*70)

# The .pt file contains state_dict and configuration
# Use torch.load() to load PyTorch checkpoints

# Get model configuration from the loaded data
config = model_data['model_config']
threshold = model_data['threshold']

print(f"\n📋 Model Configuration:")
print(f"  Input size: {config['input_size']}")
print(f"  Hidden size: {config['hidden_size']}")
print(f"  Num layers: {config['num_layers']}")
print(f"  Threshold: {threshold}")

# Create model instance (LSTMWithAttention class must be defined in your notebook)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

loaded_model = LSTMWithAttention(
    input_size=config['input_size'],
    hidden_size=config['hidden_size'],
    output_size=config['output_size'],
    num_layers=config['num_layers'],
    dropout=config['dropout']
).to(device)

# Load the saved weights
loaded_model.load_state_dict(model_data['model_state_dict'])
loaded_model.eval()  # Set to evaluation mode

print("✓ Model loaded and ready for predictions!")

In [ ]:

# ============================================================================
# STEP 2: MAKE PREDICTIONS ON TEST DATA
# ============================================================================

print("\n" + "="*70)
print("MAKING PREDICTIONS")
print("="*70)

# Option A: Predict on existing test dataset
if 'test_dataset' in globals():
  print("\n📊 Predicting on test dataset...")

  # Get a sample
  sample_sequence, sample_label = test_dataset[0]
  sample_sequence = sample_sequence.unsqueeze(0).to(device)

  with torch.no_grad():
      prediction = loaded_model(sample_sequence)
      prob = prediction.cpu().item()
      predicted_class = 'Churn' if prob > threshold else 'No Churn'
      actual_class = 'Churn' if sample_label.item() == 1 else 'No Churn'

  print(f"\n🔮 Sample Prediction:")
  print(f"  Churn Probability: {prob:.4f}")
  print(f"  Predicted: {predicted_class}")
  print(f"  Actual: {actual_class}")
  print(f"  Correct: {'✓' if predicted_class == actual_class else '✗'}")


In [ ]:
predictions = []  # Create the list first!
for i in range(min(10, len(test_dataset))):
  sequence, label = test_dataset[i]
  sequence = sequence.unsqueeze(0).to(device)

  with torch.no_grad():
      prob = loaded_model(sequence).cpu().item()
      pred_class = 1 if prob > threshold else 0

  predictions.append({
      'Index': i,
      'Actual': int(label.item()),
      'Probability': f"{prob:.4f}",
      'Predicted': pred_class,
      'Correct': '✓' if pred_class == label.item() else '✗'
  })

import pandas as pd
pred_df = pd.DataFrame(predictions)
print("\n" + pred_df.to_string(index=False))

In [ ]:
# ============================================================================
# GENERATE ROC CURVE AND AUC
# ============================================================================

print("\n" + "="*70)
print("GENERATING ROC CURVE AND AUC")
print("="*70)

if 'test_dataset' in globals():
  print("\n📊 Evaluating model on test dataset...")

  # Collect all predictions and true labels
  all_probabilities = []
  all_labels = []

  loaded_model.eval()
  with torch.no_grad():
      for i in range(len(test_dataset)):
          sequence, label = test_dataset[i]
          sequence = sequence.unsqueeze(0).to(device)

          # Get prediction probability
          prob = loaded_model(sequence).cpu().item()

          all_probabilities.append(prob)
          all_labels.append(int(label.item()))

  # Convert to numpy arrays
  y_true = np.array(all_labels)
  y_scores = np.array(all_probabilities)

  # Calculate ROC curve
  fpr, tpr, thresholds = roc_curve(y_true, y_scores)
  roc_auc = auc(fpr, tpr)

  print(f"\n📊 Results:")
  print(f"  Total samples: {len(y_true)}")
  print(f"  AUC-ROC Score: {roc_auc:.4f}")

  # Calculate metrics at default threshold
  y_pred = (y_scores > threshold).astype(int)
  from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

  accuracy = accuracy_score(y_true, y_pred)
  precision = precision_score(y_true, y_pred, zero_division=0)
  recall = recall_score(y_true, y_pred, zero_division=0)
  f1 = f1_score(y_true, y_pred, zero_division=0)

  print(f"\n📈 Performance at threshold {threshold:.2f}:")
  print(f"  Accuracy:  {accuracy:.4f}")
  print(f"  Precision: {precision:.4f}")
  print(f"  Recall:    {recall:.4f}")
  print(f"  F1 Score:  {f1:.4f}")


In [ ]:
# ========================================================================
# PLOT ROC CURVE
# ========================================================================

fig, ax = plt.subplots(figsize=(8, 6))

# Plot ROC curve
ax.plot(fpr, tpr, color='#2E86AB', linewidth=2.5,
      label=f'ROC Curve (AUC = {roc_auc:.4f})')

# Plot diagonal (random classifier)
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, alpha=0.5,
      label='Random Classifier (AUC = 0.50)')

# Mark the operating point (at threshold)
# Find the point on ROC curve closest to threshold
threshold_idx = np.argmin(np.abs(thresholds - threshold))
ax.plot(fpr[threshold_idx], tpr[threshold_idx], 'ro', markersize=10,
      label=f'Operating Point (threshold={threshold:.2f})')

# Styling
ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
ax.set_title('ROC Curve - Churn Prediction Model', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3, linewidth=0.5)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])

# Add text box with key metrics
textstr = f'AUC: {roc_auc:.4f}\nAccuracy: {accuracy:.4f}\nPrecision: {precision:.4f}\nRecall: {recall:.4f}\nF1: {f1:.4f}'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
ax.text(0.6, 0.15, textstr, transform=ax.transAxes, fontsize=10,
      verticalalignment='top', bbox=props)

plt.tight_layout()
plt.show()

print("\n✓ ROC Curve plotted!")

In [ ]:
# ========================================================================
# ADDITIONAL: PRECISION-RECALL CURVE
# ========================================================================

from sklearn.metrics import precision_recall_curve

precision_vals, recall_vals, pr_thresholds = precision_recall_curve(y_true, y_scores)

fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(recall_vals, precision_vals, color='#A23B72', linewidth=2.5,
      label='PR Curve')

# Baseline (proportion of positive class)
baseline = np.sum(y_true) / len(y_true)
ax.plot([0, 1], [baseline, baseline], 'k--', linewidth=1.5, alpha=0.5,
      label=f'Baseline (Churn Rate = {baseline:.2%})')

ax.set_xlabel('Recall', fontsize=12, fontweight='bold')
ax.set_ylabel('Precision', fontsize=12, fontweight='bold')
ax.set_title('Precision-Recall Curve - Churn Prediction Model',
          fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3, linewidth=0.5)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])

plt.tight_layout()
plt.show()

print("✓ Precision-Recall Curve plotted!")